# Download data from Amazon Web Server (AWS) bucket

This code snippet provides an example of **accessing and downloading data from AWS.** In this example, we download data from the NOAA-GMU VIIRS Flood Detection product. 
* You can read more about the product here: https://www.star.nesdis.noaa.gov/jpss/images/Harvey/show/VIIRSFloodDetectionMapQuickGuide.pdf.
* You can access the data via the AWS S3 explorer here: https://noaa-jpss.s3.amazonaws.com/index.html#JPSS_Blended_Products/VFM_1day_GLB/NetCDF/.

One type of general workflow for **downloading data from AWS** involves the following steps:

* Register for AWS to obtain your access key and secreet access key, that collectively act like username and password credentials.
* Identify the AWS bucket where the data is stored. This is usually composed of two pieces: 1) the bucket name, and 2) the prefix.
* Initialize a S3 client that establishes access to AWS.
* Generate a response from the bucket using the S3 client and extract the file download links.
* Download the file links of interest.

The main library we'll need to do this is `boto3` (read the [docs](https://docs.aws.amazon.com/boto3/latest/). You can read more about downloading files in Python from AWS here: https://docs.aws.amazon.com/boto3/latest/guide/s3-example-download-file.html

In [16]:
# Libraries
import boto3
from pathlib import Path

## Establish access to the AWS bucket

You will need to:
* Enter your own `access_key` and `secret_key`
* Update the `bucket_name` and `prefix` for the dataset you want to download

In [21]:
# Define your AWS access key and secret access key
access_key='<YOURACCESSKEY>' # MODIFY
secret_key='<YOURSECRETACCESSKEY>' # MODIFY

In [20]:
# Initialize an S3 client
s3 = boto3.client('s3', aws_access_key_id=access_key, aws_secret_access_key=secret_key)
s3

In [11]:
# Define the bucket - like the root of the data
bucket_name = 'noaa-jpss' # MODIFY

# Define the prefix - specifying the specific product, format, year, month, day
# You will likely need to modify this to fit the structure of the data you want to download
prefix = 'JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/' # MODIFY

# Create an index URL to check the data - might not work necessarily, if not don't worry
index_url = f'https://{bucket_name}.s3.amazonaws.com/index.html#{prefix}'

# Click the link to visit the bucket using AWS S3 explorer
print(index_url)

https://noaa-jpss.s3.amazonaws.com/index.html#JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/


## Get a list of files to download

In [15]:
# List files using s3.list_objects_v2()
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
# response # this will be a big json / dictionary containing a lot of info

# Get the download file links from the contents key of the response dictionary
if 'Contents' in response:
    files = [item['Key'] for item in response['Contents']]
print(f'Total files: {len(files)}')
print('\nFirst five files:')
files[:5]

Total files: 134

First five files:


['JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/VIIRS-Flood-1day-GLB001_v1r0_blend_s201901010105150_e201901012357570_c202207270019187.nc',
 'JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/VIIRS-Flood-1day-GLB002_v1r0_blend_s201901012034320_e201901012356310_c202207270019302.nc',
 'JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/VIIRS-Flood-1day-GLB003_v1r0_blend_s201901011943200_e201901012306430_c202207270019417.nc',
 'JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/VIIRS-Flood-1day-GLB004_v1r0_blend_s201901011852060_e201901012215310_c202207270019532.nc',
 'JPSS_Blended_Products/VFM_1day_GLB/NetCDF/2019/01/01/VIIRS-Flood-1day-GLB005_v1r0_blend_s201901011802200_e201901012124190_c202207270020047.nc']

## Download files

Before downloading, decide which files you want to download and where you want to download them to.

In this example, each file is a geographic area on the same date (2019/01/01): GLB001, GLB002, GLB003, etc. are global tiles. Your data you want to download will likely have a different structure.

The below code downloads all the files we obtained from the query. However, you could simply download a single file or a subset of the files by first inspecting the files in the variable `files`.

In [19]:
# Define your download path - where you want to save the downloaded files
downloadPath = Path('data/downloads/VFM_1day_GLB') # MODIFY
print(downloadPath)

data/downloads/VFM_1day_GLB


In [17]:
# Download all files returned from the response query
for file_key in files:

    # Create the file path and name - you may want to change the file naming convention
    file_name = downloadPath/(file_key.split('/')[-1])
    print(f'Saving to: {file_name}')

    # Download using download_file()
    s3.download_file(bucket_name, file_key, file_name) # (bucket, file link we got earlier, output filepath/name)
    print('Downloaded')

Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB001_v1r0_blend_s201901010105150_e201901012357570_c202207270019187.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB002_v1r0_blend_s201901012034320_e201901012356310_c202207270019302.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB003_v1r0_blend_s201901011943200_e201901012306430_c202207270019417.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB004_v1r0_blend_s201901011852060_e201901012215310_c202207270019532.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB005_v1r0_blend_s201901011802200_e201901012124190_c202207270020047.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB006_v1r0_blend_s201901011711080_e201901012033050_c202207270020165.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB007_v1r0_blend_s201901011619540_e201901011852050_c202207270020275.nc
Saving to: data/downloads/VFM_1day_GLB/VIIRS-Flood-1day-GLB008_v1r0_blend_s201901011438540_e201901011802